In [20]:
import rioxarray as rxr
import os
import dask.array as da
from shapely.geometry import mapping
import geopandas as gpd

# Paths
#ndvi_path = "Pre_aligned_rasters/NDVI_2024SummerMosaic_final.tif"
#rasters_to_align = {
#    "svf": "Pre_aligned_rasters/svf_merged.tif",
#    "albedo": "Pre_aligned_rasters/Albedo_2024SummerMosaic_final.tif",
#    "landcover": "Pre_aligned_rasters/ESA_WorldCover_2021_final.tif",
#    "ndbi": "Pre_aligned_rasters/NDBI_2024SummerMosaic_final.tif",
#    "ndmi": "Pre_aligned_rasters/DurhamNDMI_JA2024.tif",
#    "road_dist": "Pre_aligned_rasters/distance_to_main_roads_10m.tif"
#}

ndvi_path = "Filled_Rasters/ndvi_filled.tif"
rasters_to_align = {
    "svf": "Filled_Rasters/svf_filled.tif",
    "albedo": "Filled_Rasters/albedo_filled.tif",
    "landcover": "Filled_Rasters/landcover_filled.tif",
    "ndbi": "Filled_Rasters/ndbi_filled.tif",
    "ndmi": "Filled_Rasters/ndmi_filled.tif",
    "road_dist": "Filled_Rasters/road_dist_filled.tif"
}


out_dir = "Aligned_Rasters"
os.makedirs(out_dir, exist_ok=True)

# Optional shapefile clipping
boundary_shp = "Durham_Boundary/extent.shp"
boundary = gpd.read_file(boundary_shp)
boundary = boundary.to_crs(ndvi.rio.crs)
geom = [mapping(boundary.union_all())]

# Load NDVI as template
ndvi = rxr.open_rasterio(ndvi_path, masked=True, chunks={'x':1024,'y':1024})
ndvi = ndvi.rio.write_crs(ndvi.rio.crs)

# handle landcover specially - want to create a binary to match the ESA_50 field, where the value is 1 if urban, 0 if not 
print("Processing landcover...")
landcover = rxr.open_rasterio(rasters_to_align["landcover"], masked=True, chunks={'x': 1024, 'y': 1024})
landcover_aligned = landcover.rio.reproject_match(ndvi)
landcover_clipped = landcover_aligned.rio.clip(geom, ndvi.rio.crs, drop=True)

# mask out water (type 80) everywhere
water_mask = landcover_clipped == 80

# recode landcover to binary: 1 = urban (50), 0 = everything else
landcover_binary = (landcover_clipped == 50).astype("uint8")

# save recoded binary landcover
lc_out_fp = os.path.join(out_dir, "landcover_aligned.tif")
landcover_binary.rio.to_raster(lc_out_fp)
print(f"Saved {lc_out_fp}")

# Apply water mask to NDVI as well
ndvi_clipped = ndvi.rio.clip(geom, ndvi.rio.crs, drop=True)
ndvi_masked = ndvi_clipped.where(~water_mask)

ndvi_out_fp = os.path.join(out_dir, "ndvi_aligned.tif")
ndvi_masked.rio.to_raster(ndvi_out_fp)
print(f"Saved {ndvi_out_fp}")


# process other rasters with water masking and clipping
for name, path in rasters_to_align.items():
    if name == "landcover":
        continue  # already handled above

    print(f"Processing {name}...")
    da_in = rxr.open_rasterio(path, masked=True, chunks={'x': 1024, 'y': 1024})
    # align to NDVI grid
    da_aligned = da_in.rio.reproject_match(ndvi)
    # clip to boundary
    da_clipped = da_aligned.rio.clip(geom, ndvi.rio.crs, drop=True)
    # Apply water mask
    da_masked = da_clipped.where(~water_mask)

    # Save
    out_fp = os.path.join(out_dir, f"{name}_aligned.tif")
    da_masked.rio.to_raster(out_fp)
    print(f"Saved {out_fp}")


Processing landcover...
Saved Aligned_Rasters/landcover_aligned.tif
Saved Aligned_Rasters/ndvi_aligned.tif
Processing svf...
Saved Aligned_Rasters/svf_aligned.tif
Processing albedo...
Saved Aligned_Rasters/albedo_aligned.tif
Processing ndbi...
Saved Aligned_Rasters/ndbi_aligned.tif
Processing ndmi...
Saved Aligned_Rasters/ndmi_aligned.tif
Processing road_dist...
Saved Aligned_Rasters/road_dist_aligned.tif
